In [ ]:
import pandas as pd
import numpy as np
from sklearn.datasets import fetch_openml
from sklearn.model_selection import train_test_split, GridSearchCV, RandomizedSearchCV
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.metrics import accuracy_score, classification_report, roc_auc_score, roc_curve
from sklearn.calibration import calibration_curve
from sklearn.preprocessing import LabelEncoder
import matplotlib.pyplot as plt
import shap
import graphviz

# Cargar el Dataset Titanic

Utilizaremos el dataset Titanic disponible en OpenML a través de sklearn. Este dataset contiene información sobre los pasajeros del Titanic y si sobrevivieron o no.

In [ ]:
# Cargar el dataset Titanic
titanic = fetch_openml("titanic", version=1, as_frame=True)
df = titanic.frame

# Mostrar las primeras filas
print(df.head())
print(df.info())

In [ ]:
def calcular_metricas(model, X_train, y_train, X_test, y_test):
    """
    Calcula métricas ROC AUC y muestra curvas ROC y de calibración para conjuntos de train y test.

    Parameters:
    model: Modelo entrenado
    X_train, y_train: Datos de entrenamiento
    X_test, y_test: Datos de prueba
    """
    # Predicciones de probabilidad
    y_train_prob = model.predict_proba(X_train)[:, 1]
    y_test_prob = model.predict_proba(X_test)[:, 1]

    # Calcular AUC
    auc_train = roc_auc_score(y_train, y_train_prob)
    auc_test = roc_auc_score(y_test, y_test_prob)

    print(f"ROC AUC Train: {auc_train:.4f}")
    print(f"ROC AUC Test: {auc_test:.4f}")
    print(f"Diferencia (Train - Test): {auc_train - auc_test:.4f}")

    # Visualizar curvas ROC
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))

    # ROC Train
    fpr_train, tpr_train, _ = roc_curve(y_train, y_train_prob)
    ax1.plot(fpr_train, tpr_train, label=f"Train AUC = {auc_train:.3f}")
    ax1.plot([0, 1], [0, 1], "k--")
    ax1.set_xlabel("False Positive Rate")
    ax1.set_ylabel("True Positive Rate")
    ax1.set_title("ROC Curve - Train")
    ax1.legend()

    # ROC Test
    fpr_test, tpr_test, _ = roc_curve(y_test, y_test_prob)
    ax2.plot(fpr_test, tpr_test, label=f"Test AUC = {auc_test:.3f}")
    ax2.plot([0, 1], [0, 1], "k--")
    ax2.set_xlabel("False Positive Rate")
    ax2.set_ylabel("True Positive Rate")
    ax2.set_title("ROC Curve - Test")
    ax2.legend()

    plt.tight_layout()
    plt.show()

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

    # Calibración Train
    prob_true_train, prob_pred_train = calibration_curve(
        y_train, y_train_prob, n_bins=10
    )
    ax1.plot(prob_pred_train, prob_true_train, marker="o", label="Modelo")
    ax1.plot([0, 1], [0, 1], "k--", label="Perfectamente calibrado")
    ax1.set_xlabel("Probabilidad Predicha")
    ax1.set_ylabel("Fracción Positiva Real")
    ax1.set_title("Curva de Calibración - Train")
    ax1.legend()
    ax1.grid(True, alpha=0.3)

    # Calibración Test
    prob_true_test, prob_pred_test = calibration_curve(y_test, y_test_prob, n_bins=10)
    ax2.plot(prob_pred_test, prob_true_test, marker="o", label="Modelo")
    ax2.plot([0, 1], [0, 1], "k--", label="Perfectamente calibrado")
    ax2.set_xlabel("Probabilidad Predicha")
    ax2.set_ylabel("Fracción Positiva Real")
    ax2.set_title("Curva de Calibración - Test")
    ax2.legend()
    ax2.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.show()

    return auc_train, auc_test

# Preprocesar los Datos

En esta sección, manejaremos los valores faltantes, codificaremos las variables categóricas y dividiremos el dataset en conjuntos de entrenamiento y prueba.

In [ ]:
# Seleccionar características relevantes
features = ["pclass", "sex", "age", "sibsp", "parch", "fare", "embarked"]
target = "survived"

# Manejar valores faltantes
df["age"].fillna(df["age"].median(), inplace=True)
df["fare"].fillna(df["fare"].median(), inplace=True)
df["embarked"].fillna(df["embarked"].mode()[0], inplace=True)

# Codificar variables categóricas
le_sex = LabelEncoder()
df["sex"] = le_sex.fit_transform(df["sex"])

le_embarked = LabelEncoder()
df["embarked"] = le_embarked.fit_transform(df["embarked"])

# Convertir survived a int
df["survived"] = df["survived"].astype(int)

# Dividir en X e y
X = df[features]
y = df[target]

# Dividir en train y test
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print("Tamaño del conjunto de entrenamiento:", X_train.shape)
print("Tamaño del conjunto de prueba:", X_test.shape)

# Entrenar un Clasificador de Árbol de Decisión

Entrenaremos un árbol de decisión básico en el conjunto de entrenamiento.

In [ ]:
# Entrenar el modelo
clf = DecisionTreeClassifier(random_state=42)
clf.fit(X_train, y_train)

# Predecir en el conjunto de prueba
y_pred = clf.predict(X_test)

# Evaluar el modelo
print("Precisión en el conjunto de prueba:", accuracy_score(y_test, y_pred))
print("\nReporte de clasificación:\n", classification_report(y_test, y_pred))

In [ ]:
auc_train_baseline, auc_test_baseline = calcular_metricas(
    clf, X_train, y_train, X_test, y_test
)

# Visualizar el Árbol de Decisión

Utilizaremos la función plot_tree de sklearn para representar gráficamente el árbol de decisión entrenado.

In [ ]:
# Visualizar el árbol (limitado a profundidad 3 para claridad)
plt.figure(figsize=(20, 10))
plot_tree(
    clf,
    feature_names=features,
    class_names=["No sobrevivió", "Sobrevivió"],
    filled=True,
    rounded=True,
    max_depth=3,
)
plt.show()

# Calcular la Importancia de Características

Los árboles de decisión proporcionan una medida de importancia de características basada en cómo contribuyen a reducir la impureza.

In [ ]:
# Importancia de características
importances = clf.feature_importances_
feature_importance_df = pd.DataFrame(
    {"Característica": features, "Importancia": importances}
)
feature_importance_df = feature_importance_df.sort_values(
    "Importancia", ascending=False
)

print(feature_importance_df)

# Visualizar
plt.figure(figsize=(10, 6))
plt.barh(feature_importance_df["Característica"], feature_importance_df["Importancia"])
plt.xlabel("Importancia")
plt.title("Importancia de Características en el Árbol de Decisión")
plt.gca().invert_yaxis()
plt.show()

# Ilustrar la Importancia de Características con SHAP

SHAP (SHapley Additive exPlanations) proporciona una forma más precisa de explicar las predicciones del modelo.

In [ ]:
# Resumen de importancia de características, asegurando que todas se muestren
# Para un clasificador binario, shap_values.values a menudo es 3D (samples, features, outputs).
# Seleccionamos la segunda dimensión de salida (índice 1) para la clase 'Sobrevivió'.

# Usar SHAP para explicar el modelo
explainer = shap.Explainer(clf)
shap_values = explainer(X_train)[:, :, 1] # Seleccionar SHAP values para la clase 'Sobrevivió'


In [ ]:
# Para obtener un gráfico interactivo, utilizamos shap.plots.force
# Se recomienda usar los valores SHAP de una sola instancia o un rango para visualizar.
# Por ejemplo, para la primera instancia:
shap.initjs() # Inicializa la visualización JS en Colab
shap.plots.force(base_value = explainer.expected_value[1], shap_values=shap_values.values, features=X_train,)

In [ ]:
# Importance plot
shap.plots.bar(shap_values)

In [ ]:
shap.plots.beeswarm(shap_values)

In [ ]:
shap.plots.violin(shap_values)

In [ ]:
# Explicaciones para una observación en concreto
id_muestra = 0  # Cámbialo para explorar diferentes observaciones
shap.plots.waterfall(shap_values[0])

# Realizar Grid Search para Ajuste de Hiperparámetros

Grid Search (búsqueda en cuadrícula) prueba todas las combinaciones posibles de hiperparámetros especificados.

La clase `GridSearchCV` combina Grid search con validación cruzada (Cross Validation).

La validación cruzada (Cross Validation) es una técnica estadística utilizada para evaluar el rendimiento de un modelo de machine learning. Consiste en dividir el conjunto de datos en varios subconjuntos llamados "folds". Por ejemplo, en una validación cruzada de k=5 folds:

1. El conjunto de datos se divide en 5 partes iguales.
2. El modelo se entrena 5 veces, utilizando cada vez 1 fold distinto para la validación y los otros 4 folds para el entrenamiento y
3. Se calcula la métrica de rendimiento (como la precisión) en cada iteración.
4. El rendimiento final es el promedio de las métricas obtenidas en las 5 iteraciones.

Esta técnica proporciona una estimación más robusta y confiable del rendimiento del modelo en datos no vistos, ayudando a prevenir el sobreajuste (overfitting) al evaluar el modelo en múltiples particiones de los datos.

In [ ]:
# Definir los parámetros para la búsqueda
param_grid = {
    "max_depth": [3, 5, 7, None],
    "min_samples_split": [2, 5, 10],
    "min_samples_leaf": [1, 2, 4],
}

# Crear el modelo base
clf_base = DecisionTreeClassifier(random_state=42)

# Búsqueda
grid_search = GridSearchCV(
    estimator=clf_base, param_grid=param_grid, cv=5, scoring="accuracy", n_jobs=-1
)
grid_search.fit(X_train, y_train)

print("Mejores parámetros (Grid Search):", grid_search.best_params_)
print("Mejor puntuación (Grid Search):", grid_search.best_score_)

# Evaluar el mejor modelo
best_clf_grid = grid_search.best_estimator_
y_pred_grid = best_clf_grid.predict(X_test)
print("Precisión en test con Grid Search:", accuracy_score(y_test, y_pred_grid))

# Realizar Búsqueda Aleatoria para Ajuste de Hiperparámetros

La búsqueda aleatoria prueba un número limitado de combinaciones aleatorias de hiperparámetros, lo que puede ser más eficiente que la búsqueda en cuadrícula.

In [ ]:
# Definir las distribuciones de parámetros para la búsqueda aleatoria
param_dist = {
    "max_depth": [3, 5, 7, 10, None],
    "min_samples_split": np.arange(2, 20),
    "min_samples_leaf": np.arange(1, 10),
}

# Búsqueda aleatoria
random_search = RandomizedSearchCV(
    estimator=clf_base,
    param_distributions=param_dist,
    n_iter=50,
    cv=5,
    scoring="accuracy",
    random_state=42,
    n_jobs=-1,
)
random_search.fit(X_train, y_train)

print("Mejores parámetros (Random Search):", random_search.best_params_)
print("Mejor puntuación (Random Search):", random_search.best_score_)

# Evaluar el mejor modelo
best_clf_random = random_search.best_estimator_
y_pred_random = best_clf_random.predict(X_test)
print("Precisión en test con Random Search:", accuracy_score(y_test, y_pred_random))

# Optimización de Hiperparámetros guiada con Optuna
Optuna es una librería para optimizar automáticamente los hiperparámetros de los modelos. Hace pruebas con valores elegidos teniendo en cuenta las pruebas anteriores, de manera que hace combinaciones más "inteligentes" que las que produce directamente de un Random Search.

In [ ]:
!pip install optuna
import optuna
def objective(trial):
    """
    Define the objective function for Optuna to optimize.
    This function suggests hyperparameters, trains a GradientBoostingClassifier,
    and returns the ROC AUC score on the test set.
    """
    max_depth = trial.suggest_int('max_depth', 3, 10)
    min_samples_leaf = trial.suggest_int('min_samples_leaf', 1, 50)
    min_samples_split= trial.suggest_int('min_samples_split', 2, 20)
    model = DecisionTreeClassifier(
        max_depth=max_depth,
        min_samples_leaf=min_samples_leaf,
        min_samples_split=min_samples_split,
    )

    model.fit(X_train, y_train)

    # Calculate ROC AUC on the test set
    y_test_prob = model.predict_proba(X_test)[:, 1]
    auc_test = roc_auc_score(y_test, y_test_prob)

    return auc_test

print("Objective function 'objective' defined successfully.")

In [ ]:
study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=50, show_progress_bar=True)

print("Número de pruebas terminadas: {}".format(len(study.trials)))
print("Mejor prueba:")
trial = study.best_trial

print("  Valor (ROC AUC): {:.4f}".format(trial.value))
print("  Mejores hiperparámetros:")
for key, value in trial.params.items():
    print("    {}: {}".format(key, value))

In [ ]:
# Visualizar todas las pruebas
optuna.visualization.plot_optimization_history(study)